# Experiment Log — 13 Experiments, MRR Progression

Loads all benchmark results, shows progression chart and colored experiment table.

**Run with:** `$VENV_GTE -m jupyter notebook`

In [ ]:
import json, glob, os

# Manual experiment data (from recoll_experiments.md)
experiments = [
    {'#': 0, 'Change': 'Baseline', 'MRR': 0.605, 'Delta': 0, 'Kept': '-', 'Category': 'baseline'},
    {'#': 1, 'Change': 'Fix rent/salary domain (μισθ→μισθωσ)', 'MRR': 0.610, 'Delta': 0.005, 'Kept': 'Yes', 'Category': 'bm25'},
    {'#': 2, 'Change': 'ANY-mode (-o) query variant', 'MRR': 0.610, 'Delta': 0.000, 'Kept': 'No', 'Category': 'bm25'},
    {'#': 3, 'Change': 'Wildcard filename:stem*', 'MRR': 0.609, 'Delta': -0.001, 'Kept': 'No', 'Category': 'bm25'},
    {'#': 4, 'Change': 'Greek stopwords (~70 words)', 'MRR': 0.616, 'Delta': 0.006, 'Kept': 'Yes', 'Category': 'bm25'},
    {'#': 5, 'Change': 'Remove λογαριασμ from banking', 'MRR': 0.616, 'Delta': 0.000, 'Kept': 'No', 'Category': 'bm25'},
    {'#': 6, 'Change': 'Expand filename: anchors 4→6', 'MRR': 0.616, 'Delta': 0.000, 'Kept': 'No', 'Category': 'bm25'},
    {'#': 7, 'Change': 'Claude subscription query planning', 'MRR': 0.718, 'Delta': 0.102, 'Kept': 'Yes', 'Category': 'pipeline'},
    {'#': 8, 'Change': 'dir: scope from plan anchors', 'MRR': 0.726, 'Delta': 0.008, 'Kept': 'Yes', 'Category': 'pipeline'},
    {'#': 9, 'Change': 'Merge Claude + heuristic plans', 'MRR': 0.610, 'Delta': -0.116, 'Kept': 'No', 'Category': 'pipeline'},
    {'#': 10, 'Change': 'RRF vector_weight=0.8', 'MRR': 0.718, 'Delta': -0.103, 'Kept': 'No', 'Category': 'vector'},
    {'#': 11, 'Change': 'RRF vector_weight=0.3', 'MRR': 0.692, 'Delta': -0.026, 'Kept': 'No', 'Category': 'vector'},
    {'#': 12, 'Change': 'RRF vector_weight=0.15', 'MRR': 0.686, 'Delta': -0.006, 'Kept': 'No', 'Category': 'vector'},
    {'#': 13, 'Change': 'vector_augment (append-only)', 'MRR': 0.862, 'Delta': 0.041, 'Kept': 'Yes', 'Category': 'vector'},
]

print(f'{len(experiments)} experiments loaded')

In [ ]:
# Colored experiment table
import pandas as pd

df = pd.DataFrame(experiments)

cat_colors = {
    'baseline': '#F5F5F5',
    'bm25':     '#E3F2FD',   # blue
    'pipeline': '#F3E5F5',   # purple
    'vector':   '#E8F5E9',   # green
}

kept_colors = {
    'Yes': 'font-weight: bold',
    'No':  'color: #999; text-decoration: line-through',
    '-':   '',
}

def style_row(row):
    bg = cat_colors.get(row['Category'], '')
    extra = kept_colors.get(row['Kept'], '')
    base = f'background-color: {bg}'
    styles = [base] * len(row)
    # Highlight the Delta column
    delta_idx = list(row.index).index('Delta')
    if row['Delta'] > 0.01:
        styles[delta_idx] += '; color: #2E7D32; font-weight: bold'
    elif row['Delta'] < -0.01:
        styles[delta_idx] += '; color: #C62828; font-weight: bold'
    return styles

styled = (df.style
    .apply(style_row, axis=1)
    .format({'MRR': '{:.3f}', 'Delta': '{:+.3f}'})
    .set_caption('Experiment Log — Blue=BM25, Purple=Pipeline, Green=Vector')
    .hide(subset=['Category'], axis=1)
)
display(styled)

In [ ]:
# MRR Progression Chart
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 6))

# Track cumulative accepted MRR
accepted_mrr = [experiments[0]['MRR']]  # baseline
for exp in experiments[1:]:
    if exp['Kept'] == 'Yes':
        accepted_mrr.append(exp['MRR'])
    else:
        accepted_mrr.append(accepted_mrr[-1])  # stays at previous

cat_plot_colors = {'baseline': '#9E9E9E', 'bm25': '#2196F3', 'pipeline': '#9C27B0', 'vector': '#4CAF50'}

# Plot each experiment as a dot
for i, exp in enumerate(experiments):
    color = cat_plot_colors[exp['Category']]
    marker = 'o' if exp['Kept'] in ('Yes', '-') else 'x'
    size = 120 if exp['Kept'] == 'Yes' else 60
    ax.scatter(i, exp['MRR'], c=color, s=size, marker=marker, zorder=5,
              edgecolors='black' if exp['Kept'] == 'Yes' else 'none', linewidths=1.5)

# Accepted MRR line
ax.plot(range(len(accepted_mrr)), accepted_mrr, 'k--', alpha=0.4, label='Accepted MRR')

# Annotate key experiments
for i, exp in enumerate(experiments):
    if exp['Kept'] == 'Yes' and exp['Delta'] > 0.005:
        ax.annotate(f"#{exp['#']}: {exp['Change'][:25]}\n+{exp['Delta']:.3f}",
                   (i, exp['MRR']), textcoords='offset points',
                   xytext=(10, 10), fontsize=7, alpha=0.8,
                   arrowprops=dict(arrowstyle='->', alpha=0.5))
    elif exp['Delta'] < -0.05:
        ax.annotate(f"#{exp['#']}: {exp['Change'][:20]}\n{exp['Delta']:.3f}",
                   (i, exp['MRR']), textcoords='offset points',
                   xytext=(10, -20), fontsize=7, color='red', alpha=0.8)

ax.set_xlabel('Experiment #')
ax.set_ylabel('MRR')
ax.set_title('MRR Progression — 13 Experiments')
ax.set_xticks(range(len(experiments)))
ax.set_xticklabels([str(e['#']) for e in experiments])
ax.set_ylim(0.55, 0.90)
ax.grid(axis='y', alpha=0.3)

# Legend
patches = [
    mpatches.Patch(color='#2196F3', label='BM25'),
    mpatches.Patch(color='#9C27B0', label='Pipeline/Claude'),
    mpatches.Patch(color='#4CAF50', label='Vector/FAISS'),
]
ax.legend(handles=patches, loc='upper left')

plt.tight_layout()
plt.savefig('mrr_progression.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: mrr_progression.png')

In [ ]:
# Load actual benchmark JSON files for deeper analysis
bench_dir = os.path.join(os.path.dirname(os.getcwd()), 'benchmark_results')
bench_files = sorted(glob.glob(f'{bench_dir}/benchmark_*.json'))
print(f'Found {len(bench_files)} benchmark result files:')
for f in bench_files:
    name = os.path.basename(f)
    size = os.path.getsize(f) / 1024
    print(f'  {name} ({size:.0f} KB)')

## Key Takeaways

- **4 accepted, 9 reverted** — majority of ideas hurt or had no effect
- **Claude planning (#7)** = single biggest win (+0.102 MRR)
- **RRF fusion (#10-12)** always hurts regardless of weight
- **vector_augment (#13)** = correct hybrid strategy (append-only, don't re-rank)
- **Merging plans (#9)** was the biggest regression (-0.116)